In [176]:
import pandas_ta as pta
import numpy as np
import pandas as pd
import mplfinance as mpf
from tabulate import tabulate
from utils.KrakenHistoricalData import KrakenHistoricalData
from ggTrader.Signals import Signals
import matplotlib.pyplot as plt
import seaborn as sns


In [177]:
# Ticker
symbols = ["BTC", "ETH"]
interval = "4h"

# Time Range

end = pd.to_datetime("2025-06-30").tz_localize('UTC')
start = end - pd.Timedelta(days=30 * 6)

k = KrakenHistoricalData()

df_multi = k.get_ohlcv_df(symbols, interval=interval)

print(f"\nMultiIndex")
print(df_multi.info())

# select all close
print(df_multi.xs('close', axis=1, level=1).head())

# select only BTC
print(df_multi.xs('BTC', axis=1, level=0).head())

# list of tickers
print(df_multi.columns.levels[0].tolist())


MultiIndex
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5472 entries, 2023-01-01 00:00:00+00:00 to 2025-06-30 20:00:00+00:00
Freq: 4h
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   (BTC, open)    5472 non-null   float32
 1   (BTC, high)    5472 non-null   float32
 2   (BTC, low)     5472 non-null   float32
 3   (BTC, close)   5472 non-null   float32
 4   (BTC, volume)  5472 non-null   float64
 5   (BTC, trades)  5472 non-null   Int64  
 6   (BTC, base)    5472 non-null   object 
 7   (BTC, quote)   5472 non-null   object 
 8   (ETH, open)    5472 non-null   float32
 9   (ETH, high)    5472 non-null   float32
 10  (ETH, low)     5472 non-null   float32
 11  (ETH, close)   5472 non-null   float32
 12  (ETH, volume)  5472 non-null   float64
 13  (ETH, trades)  5472 non-null   Int64  
 14  (ETH, base)    5472 non-null   object 
 15  (ETH, quote)   5472 non-null   object 
dtypes: Int64(2), float32(8), fl

In [178]:
def process_ohlcv(df):
    data = {}
    col = ['open', 'high', 'low', 'close', 'volume']
    for c in col:
        data[c] = df.xs(c, axis=1, level=1)
    return data


data = process_ohlcv(df_multi)

print("\nProcessed Data")
for col in data.keys():
    print(f"{col}:")
    print(data[col].head())



Processed Data
open:
                                    BTC          ETH
2023-01-01 00:00:00+00:00  16528.699219  1195.000000
2023-01-01 04:00:00+00:00  16519.300781  1194.150024
2023-01-01 08:00:00+00:00  16512.400391  1192.199951
2023-01-01 12:00:00+00:00  16500.199219  1193.430054
2023-01-01 16:00:00+00:00  16550.000000  1197.280029
high:
                                    BTC          ETH
2023-01-01 00:00:00+00:00  16530.000000  1195.000000
2023-01-01 04:00:00+00:00  16542.400391  1195.000000
2023-01-01 08:00:00+00:00  16538.400391  1195.109985
2023-01-01 12:00:00+00:00  16550.000000  1197.719971
2023-01-01 16:00:00+00:00  16573.099609  1197.569946
low:
                                    BTC          ETH
2023-01-01 00:00:00+00:00  16505.199219  1193.000000
2023-01-01 04:00:00+00:00  16506.900391  1190.859985
2023-01-01 08:00:00+00:00  16490.000000  1192.199951
2023-01-01 12:00:00+00:00  16497.699219  1192.699951
2023-01-01 16:00:00+00:00  16531.199219  1193.800049
close:
      

In [179]:
# testing out how I can apply signals to the entire multiindex dataframe

def add_ticker_to_columns(ticker: str, df: pd.DataFrame) -> pd.DataFrame:
    df_cols = df.columns.tolist()
    new_cols = []
    for col in df_cols:
        new_cols.append((ticker, col.lower()))
    df.columns = pd.MultiIndex.from_tuples(new_cols)
    return df


def entry_signals(df: pd.DataFrame, chandelier_exit: pd.Series, adx_length: int = 14,
                  adx_threshold: int = 25) -> pd.DataFrame:
    signals = pd.DataFrame(index=df.index)

    # ADX
    adx = pta.adx(df['high'],
                  df['low'],
                  df['close'],
                  length=adx_length)
    signals['adx'] = adx.iloc[:, 0]
    signals['adx_d'] = adx.iloc[:, 1]
    signals['adx_dmp'] = adx.iloc[:, 2]
    signals['adx_dmn'] = adx.iloc[:, 3]
    signals['adx_signal'] = np.where(signals['adx'] > adx_threshold, True, False)
    signals['adx_bull'] = np.where(signals['adx_dmp'] > signals['adx_dmn'], True, False)
    signals['adx_signal'] = signals['adx_signal'] & signals['adx_bull']

    # PSAR
    psar = pta.psar(df['high'],
                    df['low'],
                    close=df['close'])
    signals['sar'] = psar.iloc[:, 0]
    signals['sar_signal'] = df['close'] > signals['sar']

    # entry

    # Have ADX strength, and sar is a buy and ce is not an exit
    entry_series = signals['adx_signal'] & signals['sar_signal'] & ~chandelier_exit
    # entry_series = signals['adx_signal'] & signals['sar_signal']
    signals['entry_rise'] = entry_series & (~entry_series.shift(1, fill_value=False))
    signals['entry_series'] = entry_series

    return signals


def exit_signals(df: pd.DataFrame, atr_multiplier: float = 3.0, ce_high_length: int = 22):
    signals = pd.DataFrame(index=df.index)

    # Exit: Chandelier Exit uses ATR
    ce = pta.chandelier_exit(df['high'],
                             df['low'],
                             df['close'],
                             multiplier=atr_multiplier, high_length=ce_high_length)

    signals['ce_l'] = np.where(ce.iloc[:, 2] > 0, ce.iloc[:, 0], np.nan)
    signals['ce_sh'] = np.where(ce.iloc[:, 2] < 0, ce.iloc[:, 1], np.nan)
    signals['ce_exit'] = np.where(ce.iloc[:, 2] == 1, False, True)

    exit_series = signals['ce_exit']

    signals['exit_rise'] = exit_series & (~exit_series.shift(1, fill_value=False))
    signals['exit_series'] = exit_series
    return signals


def filter_signals(signals: pd.DataFrame):
    in_pos = False
    filtered_entry = pd.Series(False, index=signals.index)
    filtered_exit = pd.Series(False, index=signals.index)

    for ts in signals.index:
        if not in_pos and signals['entry_rise'].loc[ts]:
            filtered_entry.loc[ts] = True
            in_pos = True
        elif in_pos and signals['exit_rise'].loc[ts]:
            filtered_exit.loc[ts] = True
            in_pos = False
    signals['entry_signal'] = filtered_entry
    signals['exit_signal'] = filtered_exit
    return signals


def calc_signals(df: pd.DataFrame, adx_length: int = 14, adx_threshold: int = 25) -> pd.DataFrame:
    df_signals = df.copy()
    if not isinstance(df.columns, pd.MultiIndex):
        print("Not a multiindex dataframe")
        return df_signals

    tickers = df.columns.levels[0].tolist()

    for ticker in tickers:
        df_single = df.xs(ticker, axis=1, level=0)

        exit_df = exit_signals(df_single,
                               atr_multiplier=3.0,
                               ce_high_length=22)

        entry_df = entry_signals(df_single,
                                 chandelier_exit=exit_df['ce_exit'],
                                 adx_length=adx_length,
                                 adx_threshold=adx_threshold)

        entry_exit_df = pd.concat([entry_df, exit_df], axis=1)
        entry_exit_df = filter_signals(entry_exit_df) # one entry -> one exit
        entry_exit_df = add_ticker_to_columns(ticker, entry_exit_df) # add ticker to columns, ex: (BTC, entry_signal)
        df_signals = pd.concat([df_signals, entry_exit_df], axis=1)
    return df_signals.sort_index(axis=1, level=0)


signals = calc_signals(df_multi)
print(signals.info())
print(signals['BTC'].tail())

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 5472 entries, 2023-01-01 00:00:00+00:00 to 2025-06-30 20:00:00+00:00
Freq: 4h
Data columns (total 50 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   (BTC, adx)           5459 non-null   float64
 1   (BTC, adx_bull)      5472 non-null   bool   
 2   (BTC, adx_d)         5457 non-null   float64
 3   (BTC, adx_dmn)       5459 non-null   float64
 4   (BTC, adx_dmp)       5459 non-null   float64
 5   (BTC, adx_signal)    5472 non-null   bool   
 6   (BTC, base)          5472 non-null   object 
 7   (BTC, ce_exit)       5472 non-null   bool   
 8   (BTC, ce_l)          3070 non-null   float64
 9   (BTC, ce_sh)         2389 non-null   float64
 10  (BTC, close)         5472 non-null   float32
 11  (BTC, entry_rise)    5472 non-null   bool   
 12  (BTC, entry_series)  5472 non-null   bool   
 13  (BTC, entry_signal)  5472 non-null   bool   
 14  (BTC, exit_rise)     5472 non-n